# GFQL Filter -> PageRank -> Filter CPU vs GPU Benchmark

This notebook documents a large-SNAP benchmark for comparing:

- `pandas + igraph` on CPU
- `cudf + cugraph` on GPU

The workflow is intentionally mixed: a first GFQL search/filter step, then PageRank, then a second GFQL step on the enriched graph. That lets us report both the dataframe/GFQL acceleration and the graph-algorithm acceleration.


## Chosen benchmark

Dataset: SNAP GPlus combined

Pipeline:
1. Load the graph
2. Compute node degree and keep the top `degree_quantile=0.995` as `seed_keep`
3. Run `gfql([n(seed_keep=True), e_undirected(hops=1), n()])`
4. Run PageRank with `compute_igraph(pagerank)` or `compute_cugraph(pagerank)`
5. Keep the top `pagerank_quantile=0.9995` as `core_keep`
6. Run a second one-hop GFQL expansion around those high-PageRank cores


## DGX result snapshot

Warm timings on `dgx-spark` using `graphistry/test-gpu:latest`:

- Warm CPU pipeline: `82.48s`
- Warm GPU pipeline: `4.08s`
- Warm speedup: `20.24x`

Stage medians:

| Stage | CPU | GPU | Speedup |
|---|---:|---:|---:|
| GFQL filter 1 | `47.47s` | `2.92s` | `16.26x` |
| PageRank | `22.43s` | `0.58s` | `38.91x` |
| GFQL filter 2 | `12.58s` | `0.58s` | `21.72x` |
| Warm pipeline total | `82.48s` | `4.08s` | `20.24x` |

Graph sizes:

| Stage | CPU | GPU |
|---|---:|---:|
| Full graph | `107,614 nodes / 30,494,866 edges` | `107,614 nodes / 30,494,866 edges` |
| After GFQL filter 1 | `73,010 nodes / 11,755,106 edges` | `73,010 nodes / 11,755,106 edges` |
| After GFQL filter 2 | `56,725 nodes / 1,556,371 edges` | `56,930 nodes / 1,367,990 edges` |

The final graph differs modestly because `igraph` and `cugraph` produce slightly different PageRank score distributions, so the top-quantile cutoff lands on a different boundary.


In [ ]:
# Reproduce on DGX: GPU
!python benchmarks/gfql/filter_pagerank_pipeline_cpu_gpu.py \
    --dataset gplus \
    --engine cudf \
    --degree-quantile 0.995 \
    --pagerank-quantile 0.9995 \
    --warmup 1 --runs 2 \
    --output-json plans/gfql-gpu-pagerank-benchmark/results/gplus_gpu_q995_pr9995.json


In [ ]:
# Reproduce on DGX: CPU
!python benchmarks/gfql/filter_pagerank_pipeline_cpu_gpu.py \
    --dataset gplus \
    --engine pandas \
    --degree-quantile 0.995 \
    --pagerank-quantile 0.9995 \
    --warmup 1 --runs 1 \
    --output-json plans/gfql-gpu-pagerank-benchmark/results/gplus_cpu_q995_pr9995.json


In [ ]:
import json
from pathlib import Path
summary = json.loads(Path("plans/gfql-gpu-pagerank-benchmark/results/gplus_q995_pr9995_summary.json").read_text())
summary["speedup_summary"]


## Environment

Host: `dgx-spark`

- GPU: `NVIDIA GB10`
- Driver: `580.126.09`
- RAM at measurement time: `119Gi total`, `12Gi available`

Container: `graphistry/test-gpu:latest`

- Python `3.12.12`
- pandas `2.3.3`
- cudf `25.12.00`
- cugraph `25.12.02`
- igraph `1.0.0`


## Optional next step

A stretch comparison is to load the same graph or selected filtered subgraph into Neo4j and compare an approximate Cypher/APOC/GDS analog, but that is not required for the core Graphistry CPU-vs-GPU story.
